<a href="https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory data analysis of multidimensional poverty in Bolivia

Part of the [GeoDevelopment Observatory of Bolivia](https://quarcs-lab.github.io/gdo4bolivia/).

This notebook explores how multidimensional poverty varies across Bolivia's 343 municipalities and indigenous autonomies, how it changed between the 2012 and 2024 censuses, and how it relates to population and night-time lights. The data and every analysis come from the [`geo4bolivia`](https://quarcs-lab.github.io/geo4bolivia/) package.

1. **Load the data** with `geo4bolivia`
2. **Descriptive statistics**: the centre, the spread and the extremes, in both periods
3. **Regional differences**: how poverty is distributed across departments
4. **Change, 2012-2024**: where poverty fell most
5. **Population and poverty**: are the most populous municipalities less poor?
6. **Night-time lights and poverty**: can lights from space help predict poverty?

Each exploration function ([`distribution`](https://quarcs-lab.github.io/geo4bolivia/reference/distribution.html), [`extremes`](https://quarcs-lab.github.io/geo4bolivia/reference/extremes.html), [`correlation`](https://quarcs-lab.github.io/geo4bolivia/reference/correlation.html), [`scatter`](https://quarcs-lab.github.io/geo4bolivia/reference/scatter.html), [`trend`](https://quarcs-lab.github.io/geo4bolivia/reference/trend.html)) returns a **result object**. Displayed, it shows its figure and a reading of the numbers; `.fig`, `.df`, `.gt` and the named numbers (`.corr`, `.slope`, ...) give the parts one by one. Every figure is interactive: hover over a point or a municipality to see its name.

## 0. Setup

In Google Colab the next cell installs the package (a few seconds). Locally, `uv sync` has installed it already.

In [1]:
import importlib.util

if importlib.util.find_spec("geo4bolivia") is None:
    !pip install -q geo4bolivia

In [2]:
import sys
import warnings

import pandas as pd
import plotly.express as px
import plotly.io as pio

import geo4bolivia as geo

if "google.colab" not in sys.modules:
    pio.renderers.default = "notebook_connected"   # keeps the saved notebook small
warnings.filterwarnings("ignore", module="numba")   # a notice of an optional library

## 1. Load the data

| Variable | Dataset | Periods | Meaning |
| --- | --- | --- | --- |
| `sdg1_2_mpi` | `sdg01` | 2012, 2024 | Multidimensional Poverty Index (0–1; **higher = poorer**), from the censuses |
| `population` | `sdg` | 2012, 2024 | Census population |
| `ln_t400NTLpc` | `ntl` | 2012–2020 | Long-run trend (HP filter) of the log of night-time lights per capita |

The poverty index comes from the [Atlas de los ODS en Bolivia 2026](https://atlasods.sdsnbolivia.org/) (Andersen & Argandoña, SDSN Bolivia), which computes it from the 2012 and 2024 Population and Housing Censuses for all 343 units. Night-time lights were computed on the older 339-unit map, so they cover **333 units**: the six municipalities that have since lost territory (Colquechaca, Mizque, Porco, Tomave, San Ignacio and Santa Ana) and the four units created after 2012 have no values.

`geo.load()` returns a panel with one row per unit and year. We join the three variables into one panel; lights enter in 2012, the one year they share with poverty.

In [3]:
mpi = geo.load("sdg01", variables="sdg1_2_mpi", groups=["department"])
census = geo.load("sdg", variables="population")
lights = geo.load("ntl", variables="ln_t400NTLpc")          # 2012 to 2020

df = (mpi.merge(census, on=["geo_id", "year"])
         .merge(lights, on=["geo_id", "year"], how="left"))
df.head()

,geo_id,name,year,department,sdg1_2_mpi,population,ln_t400NTLpc
0,BO-01-01-01,Azurduy,2012,Chuquisaca,0.512,10652,11.048312
1,BO-01-01-01,Azurduy,2024,Chuquisaca,0.187,7903,NaN
2,BO-01-01-02,Tarvita,2012,Chuquisaca,0.452,14261,12.329279
3,BO-01-01-02,Tarvita,2024,Chuquisaca,0.133,16812,NaN
4,BO-01-02-01,Villa Serrano,2012,Chuquisaca,0.267,11161,12.815315


## 2. Descriptive statistics

- What do we know about the centre and dispersion of poverty, and how did they change?

`geo.describe()` summarises each variable in each year, with its label and unit from the metadata.

In [4]:
geo.describe(df).drop(columns=["variable", "unit"]).round(3)

,label,year,count,mean,std,min,p25,median,p75,max
0,Multidimensional poverty index (0-1),2012,343,0.234,0.106,0.018,0.153,0.227,0.306,0.563
1,Multidimensional poverty index (0-1),2024,343,0.063,0.044,0.003,0.030,0.053,0.083,0.296
2,Population (census),2012,343,29329.026,107270.842,509.000,5945.000,10647.000,19544.000,1444425.000
3,Population (census),2024,343,33133.915,116499.084,875.000,6691.500,12296.000,21083.000,1610982.000
4,Log night-time lights per capita (trend),2012,333,13.030,1.682,0.856,12.041,13.118,14.277,17.071
5,Log night-time lights per capita (trend),2024,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


`geo.extremes()` ranks one year and returns the units at the two ends.

In [5]:
geo.extremes(df, "sdg1_2_mpi", year=2012)

**Multidimensional poverty index (0-1)** in 2012 ranges from 0.018 (Nuestra Señora de La Paz) to 0.563 (TIOC-Raqaypampa) across the 343 units with data; the median unit has 0.227.

The 5 highest: TIOC-Raqaypampa (0.563), Poroma (0.541), Azurduy (0.512), Sena (0.507), San Pedro (0.496).

The 5 lowest: Nuestra Señora de La Paz (0.018), Huachacalla (0.026), Colcapirhua (0.039), Tarija (0.045), Oruro (0.046).

The gap between the two ends is 0.545. Extreme values deserve a second look before any analysis: they may be the most interesting units, very small units whose rates are noisy, or errors in the data.

_These are associations, not causal effects. A causal reading needs a research design._

In [6]:
geo.extremes(df, "sdg1_2_mpi", year=2024)

**Multidimensional poverty index (0-1)** in 2024 ranges from 0.003 (Nuestra Señora de La Paz) to 0.296 (TIOC-Raqaypampa) across the 343 units with data; the median unit has 0.053.

The 5 highest: TIOC-Raqaypampa (0.296), Poroma (0.223), San Pedro (0.213), El Choro (0.203), Vila Vila (0.2).

The 5 lowest: Nuestra Señora de La Paz (0.003), Colcapirhua (0.005), Trigal (0.006), Santa Cruz de la Sierra (0.006), El Alto (0.006).

The gap between the two ends is 0.293. Extreme values deserve a second look before any analysis: they may be the most interesting units, very small units whose rates are noisy, or errors in the data.

_These are associations, not causal effects. A causal reading needs a research design._

In 2012 the average municipality had an MPI of 0.234 (median 0.227, standard deviation 0.106), with values from 0.018 to 0.563. By 2024 the average had fallen to 0.063 (median 0.053, standard deviation 0.044), with values from 0.003 to 0.296. Population is very skewed: the median unit had about 12,300 inhabitants in 2024 (10,600 in 2012), while Santa Cruz de la Sierra had 1.61 million. The summary table also shows a gap: lights have 333 units in 2012 and none in 2024.

The poorest unit in both periods is TIOC-Raqaypampa (Cochabamba; 0.563 in 2012 and 0.296 in 2024), an indigenous autonomy created after 2012, followed by Poroma (Chuquisaca). Nuestra Señora de La Paz is the least poor in both periods (0.018 and 0.003); in 2024 Colcapirhua, Trigal (Santa Cruz), Santa Cruz de la Sierra and El Alto follow it.

## 3. Regional differences in poverty

- How is poverty distributed across the 343 municipalities and 9 departments, and did the pattern change?

`geo.distribution()` draws the histogram of a variable. With two periods it overlays them on common bins.

In [7]:
geo.distribution(df, "sdg1_2_mpi")

Between 2012 and 2024, the mean of **Multidimensional poverty index (0-1)** fell from 0.234 to 0.0633, and the median went from 0.227 to 0.053.

The spread across units narrowed: the standard deviation went from 0.106 to 0.0441, and the interquartile range (the middle half of the units) from 0.153 to 0.053.

**Multidimensional poverty index (0-1)** in 2024 averages 0.0633 across 343 units (median 0.053, standard deviation 0.0441). Half of the units lie between 0.03 and 0.083, and the full range runs from 0.003 to 0.296. The distribution is strongly right-skewed: a tail of high values pulls the mean above the median (skewness 1.44).

_These are associations, not causal effects. A causal reading needs a research design._

With `by=`, it compares groups in one year. A box plot shows each department's median, its middle half and its unusual municipalities:

In [8]:
box_2012 = geo.distribution(df, "sdg1_2_mpi", year=2012, by="department",
                            kind="box")
box_2012

By Department in 2012 (9 groups), the median of **Multidimensional poverty index (0-1)** is highest in **Pando** (0.336) and lowest in **Tarija** (0.139).

Units differ most within **Cochabamba** (standard deviation 0.126) and least within **Tarija** (0.0516). Where the groups' distributions barely overlap, the differences between groups are large compared with those within them.

In each box, the line is the median and the box holds the middle half of the units; points beyond the whiskers are unusually high or low units.

_These are associations, not causal effects. A causal reading needs a research design._

In [9]:
box_2024 = geo.distribution(df, "sdg1_2_mpi", year=2024, by="department",
                            kind="box")
box_2024

By Department in 2024 (9 groups), the median of **Multidimensional poverty index (0-1)** is highest in **Potosí** (0.0775) and lowest in **Tarija** (0.023).

Units differ most within **Cochabamba** (standard deviation 0.0565) and least within **Tarija** (0.0125). Where the groups' distributions barely overlap, the differences between groups are large compared with those within them.

In each box, the line is the median and the box holds the middle half of the units; points beyond the whiskers are unusually high or low units.

_These are associations, not causal effects. A causal reading needs a research design._

The table behind each figure is `.df`. Side by side, the department means and medians of the two periods:

In [10]:
cols = ["department", "n", "mean", "median", "sd", "min", "max"]
by_dep = box_2012.df[cols].merge(box_2024.df[cols], on=["department", "n"],
                                 suffixes=("_2012", "_2024"))
by_dep.sort_values("mean_2012", ascending=False).round(3)

,department,n,mean_2012,median_2012,sd_2012,min_2012,max_2012,mean_2024,median_2024,sd_2024,min_2024,max_2024
1,Chuquisaca,29,0.323,0.317,0.105,0.071,0.541,0.090,0.074,0.054,0.014,0.223
5,Pando,15,0.315,0.336,0.105,0.112,0.507,0.068,0.056,0.038,0.019,0.177
6,Potosí,42,0.279,0.270,0.117,0.051,0.496,0.083,0.078,0.052,0.012,0.213
0,Beni,20,0.250,0.227,0.078,0.145,0.460,0.075,0.062,0.037,0.035,0.168
4,Oruro,35,0.235,0.232,0.095,0.026,0.443,0.069,0.066,0.046,0.007,0.203
2,Cochabamba,48,0.230,0.222,0.126,0.039,0.563,0.070,0.056,0.056,0.005,0.296
3,La Paz,87,0.224,0.222,0.079,0.018,0.434,0.060,0.056,0.028,0.003,0.144
7,Santa Cruz,56,0.169,0.154,0.065,0.051,0.338,0.032,0.027,0.017,0.006,0.076
8,Tarija,11,0.118,0.139,0.052,0.045,0.201,0.027,0.023,0.012,0.010,0.045


How much of the differences between municipalities lies between departments? `geo.theil_decomposition()` splits Theil's inequality index into a part between groups and a part within them.

In [11]:
geo.theil_decomposition(df, "sdg1_2_mpi", by="department")

The Theil index of **Multidimensional poverty index (0-1)** splits additively into inequality **between** the 9 department groups and inequality **within** them (between + within = total, exactly).

In the latest year (2024), the total Theil index is 0.219: about 21% of it lies between department groups and 79% within them — most inequality plays out among units inside the same department.

Over the window, the between-group share stayed at about 21%.

_These are associations, not causal effects. A causal reading needs a research design._

Departments differ, but municipalities differ much more within departments than the departments differ from one another: only 21% of the Theil index lies between departments, in 2012 and in 2024. In 2012 Cochabamba ran from 0.039 to the highest value in the country (0.563, TIOC-Raqaypampa), and Potosí from 0.051 to 0.496. Pando and Chuquisaca had the highest medians (0.336 and 0.317) and Chuquisaca the highest mean (0.323); Tarija's poorest municipality scored 0.201.

By 2024 the whole distribution has shifted towards zero and narrowed: the national median is 0.053 and the interquartile range fell from 0.153 to 0.053. What remains is a long right tail (skewness 1.44). The order of the departments is broadly the same, with Chuquisaca (mean 0.090) and Potosí (0.083) the poorest, followed by Beni (0.075), and Santa Cruz (0.032) and Tarija (0.027) the least poor. Pando's mean fell the most, from 0.315 to 0.068.

## 4. Change, 2012-2024

With a list of years and a classification scheme, `geo.map()` draws the two periods side by side in the same classes (here natural breaks computed over both years), so a colour means the same on both maps.

In [12]:
geo.map(df, "sdg1_2_mpi", year=[2012, 2024], scheme="fisher_jenks")

`change=True` maps the change itself, 2024 minus 2012, in MPI points:

In [13]:
geo.map(df, "sdg1_2_mpi", change=True, scheme="fisher_jenks")

`geo.change()` gives the same numbers as a table: one row per unit with its value in each year, its change in points and in percent, and its rank in each year.

In [14]:
chg = geo.change(df, "sdg1_2_mpi")
summary = geo.describe(chg, ["sdg1_2_mpi_change", "sdg1_2_mpi_pct_change"], by=None)
summary.drop(columns=["variable", "unit"]).round(3)

,label,count,mean,std,min,p25,median,p75,max
0,Change in Multidimensional poverty index (0-1),343,-0.171,0.071,-0.402,-0.225,-0.169,-0.118,-0.015
1,% change in Multidimensional poverty index (0-1),343,-74.577,9.270,-95.669,-80.607,-75.862,-69.763,-32.979


Where did poverty fall most and least? The change is negative everywhere, so the "lowest" values are the largest falls.

In [15]:
geo.extremes(chg, "sdg1_2_mpi_change")

**Change in Multidimensional poverty index (0-1)** ranges from -0.402 (Sena) to -0.015 (Nuestra Señora de La Paz) across the 343 units with data; the median unit has -0.169.

The 5 highest: Nuestra Señora de La Paz (-0.015), Huachacalla (-0.018), Tolata (-0.031), Colcapirhua (-0.034), Tarija (-0.035).

The 5 lowest: Sena (-0.402), Tinguipaya (-0.328), Alalay (-0.327), Azurduy (-0.325), Presto (-0.324).

The gap between the two ends is 0.387. Extreme values deserve a second look before any analysis: they may be the most interesting units, very small units whose rates are noisy, or errors in the data.

_These are associations, not causal effects. A causal reading needs a research design._

Is the poverty of 2024 related to the poverty of 2012?

In [16]:
geo.scatter(chg, "sdg1_2_mpi_2012", "sdg1_2_mpi_2024", color="department")

Across 343 units, **Multidimensional poverty index (0-1), 2024** and **Multidimensional poverty index (0-1), 2012** are positively correlated (r = 0.86, a strong association): units with higher Multidimensional poverty index (0-1), 2012 tend to have higher Multidimensional poverty index (0-1), 2024.

The least-squares line has a slope of 0.359: one more unit of Multidimensional poverty index (0-1), 2012 goes with 0.359 higher Multidimensional poverty index (0-1), 2024, on average. The line accounts for 74% of the variation in Multidimensional poverty index (0-1), 2024 across units (R² = 0.74); the slope is statistically significant at the 1% level, by a test that treats the units as independent of each other.

_These are associations, not causal effects. A causal reading needs a research design._

The correlation table (`.gt`) has Pearson's coefficient above the diagonal and Spearman's rank coefficient below it:

In [17]:
geo.correlation(chg, ["sdg1_2_mpi_2012", "sdg1_2_mpi_2024", "sdg1_2_mpi_change"]).gt

GT(_tbl_data=                                            variable      A      B      C
0      A: Multidimensional poverty index (0-1), 2012          0.86  -0.95
1      B: Multidimensional poverty index (0-1), 2024   0.86         -0.65
2  C: Change in Multidimensional poverty index (0-1)  -0.96  -0.71       , _body=<great_tables._gt_data.Body object at 0x130e33910>, _boxhead=Boxhead([ColInfo(var='variable', type=<ColInfoTypeEnum.stub: 2>, column_label='variable', column_align='left', column_width=None, stub_level=0), ColInfo(var='A', type=<ColInfoTypeEnum.default: 1>, column_label='A', column_align='right', column_width=None, stub_level=None), ColInfo(var='B', type=<ColInfoTypeEnum.default: 1>, column_label='B', column_align='right', column_width=None, stub_level=None), ColInfo(var='C', type=<ColInfoTypeEnum.default: 1>, column_label='C', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x130e31bd0>, _spanners=Spanners([]), _heading=Heading(title='Pearson correlations', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x130e33890>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x130e302d0>, _source_notes=['Pearson correlations above the diagonal, Spearman rank correlations below. Observations: 343.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x130e30d90>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='ta

`geo.trend()` follows the mean of each department between the two periods; the thin grey lines are the municipalities.

In [18]:
geo.trend(df, "sdg1_2_mpi", by="department")

Between 2012 and 2024, the mean of **Multidimensional poverty index (0-1)** across 343 units fell from 0.234 to 0.0633 (a change of -0.171).

No unit with data in both years ended higher than it started.

By Department: **Tarija** fell least (-0.0914) and **Pando** fell most (-0.247). The gap between the highest and the lowest group narrowed from 0.205 to 0.0629.

The band is one standard error on each side of the mean: how precisely the average is known, not how much units differ.

_These are associations, not causal effects. A causal reading needs a research design._

Did the gaps between municipalities narrow? `geo.inequality()` measures them in relative terms, with the Gini index and the coefficient of variation (the standard deviation over the mean).

In [19]:
geo.inequality(df, "sdg1_2_mpi", measures=["gini", "cv"])

Cross-sectional inequality in **Multidimensional poverty index (0-1)** is tracked across 343 units over 2 years (2012 to 2024).

The Gini index rose from 0.254 to 0.367 between the first and last year.

The coefficient of variation rose from 0.45 to 0.696 between the first and last year.

With two years (2012 and 2024) the measures can be compared, but their trend cannot be tested: that needs three or more years.

On the Lorenz curve of 2024, the half of the units with the lowest values holds 25% of the total (32% in 2012) and the tenth with the highest values holds 25% (19%); with no inequality they would hold 50% and 10%.

_These are associations, not causal effects. A causal reading needs a research design._

Poverty fell in all 343 units: the change runs from -0.402 to -0.015 MPI points, with a mean of -0.171. In relative terms the falls are similar everywhere: the median unit lost 76% of its 2012 value, and half of the units lost between 70% and 81%. In the scatter, the least-squares line has a slope of 0.359, far below one: the higher the poverty of 2012, the larger the fall in points (the correlation between the 2012 level and the change is -0.95). The ordering persists (correlation 0.86, rank correlation 0.86), so the poorest places of 2012 are mostly still the poorest.

Absolute gaps narrowed (the standard deviation fell from 0.106 to 0.044, and the gap between the poorest and the least poor department mean from 0.205 to 0.063), but relative gaps widened: the coefficient of variation rose from 0.45 to 0.70 and the Gini index from 0.254 to 0.367, because the least poor municipalities are now very close to zero. The largest fall was in Sena (Pando, -0.402), followed by Tinguipaya (Potosí), Alalay (Cochabamba), Azurduy and Presto (Chuquisaca), all about -0.33. Four of the five smallest falls were in the places with the lowest poverty of 2012, which had little to lose (Nuestra Señora de La Paz, Huachacalla, Colcapirhua and Tarija); the fifth is Tolata (Cochabamba, -0.031). Among departments, Pando's mean fell the most (-0.247) and Tarija's the least (-0.091).

## 5. Population and poverty

- Do the most populous municipalities tend to be less poor?

Population here is the census count of the same year. The ten most and the ten least populous units in 2024:

In [20]:
size = geo.extremes(df, "population", year=2024, n=10)
size.fig

The treemap nests municipalities within departments: the area of each tile is its 2024 population and its colour its 2024 poverty (click a department to zoom in). The package has no treemap, so this figure and the next are plain Plotly Express, with `geo.labels()` naming the columns.

In [21]:
d24 = df[df["year"] == 2024]
px.treemap(d24, path=[px.Constant("Bolivia"), "department", "name"],
           values="population", color="sdg1_2_mpi", labels=geo.labels(d24),
           title="Population (area) and multidimensional poverty (colour), 2024")

The same view for 2012, as a sunburst:

In [22]:
d12 = df[df["year"] == 2012]
px.sunburst(d12, path=["department", "name"], values="population",
            color="sdg1_2_mpi", labels=geo.labels(d12),
            title="Population (angle) and multidimensional poverty (colour), 2012")

`geo.scatter()` with `log_x=True` puts population on a logarithmic axis and fits the line on its natural logarithm:

In [23]:
geo.scatter(df, "population", "sdg1_2_mpi", year=2012, color="department",
            log_x=True)

Across 343 units in 2012, **Multidimensional poverty index (0-1)** and **Population (census)** are negatively correlated (r = -0.34, a weak association): units with higher Population (census) tend to have lower Multidimensional poverty index (0-1).

The least-squares line has a slope of -0.0313: 1% more of Population (census) goes with 0.000313 lower Multidimensional poverty index (0-1), on average (Population (census) is in logs). The line accounts for 11% of the variation in Multidimensional poverty index (0-1) across units (R² = 0.11); the slope is statistically significant at the 1% level, by a test that treats the units as independent of each other.

_These are associations, not causal effects. A causal reading needs a research design._

In [24]:
geo.scatter(df, "population", "sdg1_2_mpi", year=2024, color="department",
            log_x=True)

Across 343 units in 2024, **Multidimensional poverty index (0-1)** and **Population (census)** are negatively correlated (r = -0.23, a weak association): units with higher Population (census) tend to have lower Multidimensional poverty index (0-1).

The least-squares line has a slope of -0.00903: 1% more of Population (census) goes with 9.03e-05 lower Multidimensional poverty index (0-1), on average (Population (census) is in logs). The line accounts for 5% of the variation in Multidimensional poverty index (0-1) across units (R² = 0.05); the slope is statistically significant at the 1% level, by a test that treats the units as independent of each other.

_These are associations, not causal effects. A causal reading needs a research design._

The typical municipality is not where the typical Bolivian lives. `geo.aggregate()` computes the national poverty index as a census-population-weighted mean of the municipalities (the rule the metadata declares), to compare with the simple means of section 2. The last lines measure the weight of the ten largest municipalities.

In [25]:
geo.aggregate(df, to="country", dataset="sdg01", variables="sdg1_2_mpi").round(3)

,geo_id,year,sdg1_2_mpi,n_units,coverage
0,BO,2012,0.133,343,1.0
1,BO,2024,0.033,343,1.0


In [26]:
largest = size.df.loc[size.df["end"] == "highest", "geo_id"]
top10 = d24[d24["geo_id"].isin(largest)]
share = top10["population"].sum() / d24["population"].sum()
print(f"The ten largest municipalities hold {share:.0%} of the 2024 population; "
      f"their highest MPI is {top10['sdg1_2_mpi'].max():.3f}.")

The ten largest municipalities hold 47% of the 2024 population; their highest MPI is 0.021.


More populous municipalities tend to be less poor, in both periods. In 2012 the correlation with log population is -0.34, and a tenfold increase in population goes with an MPI about 0.07 lower (0.031 × ln 10). In 2024 the relation is weaker (correlation -0.23) and the same tenfold increase goes with an MPI only 0.02 lower (0.009 × ln 10), as poverty has fallen most where it was highest. The relationship is loose: population accounts for 11% of the variation of poverty in 2012 and 5% in 2024.

Population and poverty are concentrated in different places. The ten largest municipalities hold 47% of the 2024 population and none has an MPI above 0.021, while three quarters of the units have fewer than about 21,000 people. As a result the population-weighted mean MPI is about half the simple mean: 0.133 against 0.234 in 2012, and 0.033 against 0.063 in 2024. The typical Bolivian lives in a much less poor municipality than the typical municipality suggests.

## 6. Night-time lights and poverty

- Can night-time lights help us predict poverty?

Lights are measured every year from 2012 to 2020. `geo.trend()` shows the mean of each department over those years, on the 333 units with lights:

In [27]:
geo.trend(lights, "ln_t400NTLpc", by="department")

Between 2012 and 2020, the mean of **Log night-time lights per capita (trend)** across 333 units rose from 13 to 14 (a change of +1.01).

95% of the units with data in both years ended higher than they started: the summary line describes the typical unit, not every unit.

By Department: **Beni** grew most (+1.9, from 12 to 13.9) and **Santa Cruz** grew least (+0.577). The gap between the highest and the lowest group narrowed from 2.51 to 2.02.

The band is one standard error on each side of the mean: how precisely the average is known, not how much units differ.

_These are associations, not causal effects. A causal reading needs a research design._

We compare lights with poverty in 2012, the year both were measured. The map shows lights per capita in quintiles: a few almost unlit municipalities are extreme outliers, and natural breaks would give them a class of their own. The ten units without lights are left blank. Compare it with the 2012 poverty map of section 4.

In [28]:
geo.map(df, "ln_t400NTLpc", year=2012, scheme="quantiles")

In [29]:
geo.scatter(df, "ln_t400NTLpc", "sdg1_2_mpi", year=2012, color="department")

Across 333 units in 2012, **Multidimensional poverty index (0-1)** and **Log night-time lights per capita (trend)** are negatively correlated (r = -0.54, a moderate association): units with higher Log night-time lights per capita (trend) tend to have lower Multidimensional poverty index (0-1).

The least-squares line has a slope of -0.0333: one more unit of Log night-time lights per capita (trend) goes with 0.0333 lower Multidimensional poverty index (0-1), on average. The line accounts for 29% of the variation in Multidimensional poverty index (0-1) across units (R² = 0.29); the slope is statistically significant at the 1% level, by a test that treats the units as independent of each other.

_These are associations, not causal effects. A causal reading needs a research design._

A few municipalities are almost unlit. Are they driving the result? `geo.extremes()` finds them; the rank (Spearman) correlation, below the diagonal of the table, and the correlation without them answer the question.

In [30]:
geo.extremes(df, "ln_t400NTLpc", year=2012, n=3).df

,geo_id,name,department,ln_t400NTLpc,rank,end
0,BO-09-04-01,Caraparí,Tarija,17.071051,1,highest
1,BO-07-12-01,"Colcha ""K""",Potosí,16.250179,2,highest
2,BO-08-04-07,Lagunillas,Santa Cruz,16.217480,3,highest
3,BO-03-08-01,Exaltación,Beni,6.194119,331,lowest
4,BO-06-03-02,San Lorenzo,Pando,5.467239,332,lowest
5,BO-07-15-01,Mojinete,Potosí,0.856095,333,lowest


In [31]:
geo.correlation(df, ["sdg1_2_mpi", "ln_t400NTLpc", "population"], year=2012).gt

GT(_tbl_data=                                      variable      A      B      C
0      A: Multidimensional poverty index (0-1)         -0.54  -0.27
1  B: Log night-time lights per capita (trend)  -0.60          0.20
2                       C: Population (census)  -0.33   0.36       , _body=<great_tables._gt_data.Body object at 0x132e07e10>, _boxhead=Boxhead([ColInfo(var='variable', type=<ColInfoTypeEnum.stub: 2>, column_label='variable', column_align='left', column_width=None, stub_level=0), ColInfo(var='A', type=<ColInfoTypeEnum.default: 1>, column_label='A', column_align='right', column_width=None, stub_level=None), ColInfo(var='B', type=<ColInfoTypeEnum.default: 1>, column_label='B', column_align='right', column_width=None, stub_level=None), ColInfo(var='C', type=<ColInfoTypeEnum.default: 1>, column_label='C', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x132e10250>, _spanners=Spanners([]), _heading=Heading(title='Pearson correlations, 2012', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x132e07050>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x132e06650>, _source_notes=['Pearson correlations above the diagonal, Spearman rank correlations below. Observations: 333.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x132e06c50>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_bottom_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_bottom_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_left_style=OptionsInfo(scss=True, category='table', type='value'

In [32]:
lit = df[df["ln_t400NTLpc"] > 7]
print(f"Correlation without the three almost unlit units: "
      f"{geo.scatter(lit, 'ln_t400NTLpc', 'sdg1_2_mpi', year=2012).corr:.2f}")

Correlation without the three almost unlit units: -0.57


Do the lights of 2012 also anticipate poverty in 2024, and do the latest lights (2020) relate to it? `wide=` loads one column per year, which we join to the table of changes of section 4. `geo.labels(..., own=...)` gives the new columns a label for the table.

In [33]:
lights_wide = geo.load("ntl", variables="ln_t400NTLpc", wide="ln_t400NTLpc")
later = chg.merge(lights_wide[["geo_id", "ln_t400NTLpc2012", "ln_t400NTLpc2020"]],
                  on="geo_id")
geo.labels(later, own={"ln_t400NTLpc2012": "Log lights per capita (trend), 2012",
                       "ln_t400NTLpc2020": "Log lights per capita (trend), 2020"})
geo.correlation(later, ["sdg1_2_mpi_2024", "sdg1_2_mpi_change",
                        "ln_t400NTLpc2012", "ln_t400NTLpc2020"]).gt

GT(_tbl_data=                                            variable      A      B      C  \
0      A: Multidimensional poverty index (0-1), 2024         -0.66  -0.49   
1  B: Change in Multidimensional poverty index (0-1)  -0.71          0.50   
2             C: Log lights per capita (trend), 2012  -0.58   0.56          
3             D: Log lights per capita (trend), 2020  -0.57   0.57   0.84   

       D  
0  -0.50  
1   0.52  
2   0.76  
3         , _body=<great_tables._gt_data.Body object at 0x132fbae10>, _boxhead=Boxhead([ColInfo(var='variable', type=<ColInfoTypeEnum.stub: 2>, column_label='variable', column_align='left', column_width=None, stub_level=0), ColInfo(var='A', type=<ColInfoTypeEnum.default: 1>, column_label='A', column_align='right', column_width=None, stub_level=None), ColInfo(var='B', type=<ColInfoTypeEnum.default: 1>, column_label='B', column_align='right', column_width=None, stub_level=None), ColInfo(var='C', type=<ColInfoTypeEnum.default: 1>, column_label='C', column_align='right', column_width=None, stub_level=None), ColInfo(var='D', type=<ColInfoTypeEnum.default: 1>, column_label='D', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x132fb9ed0>, _spanners=Spanners([]), _heading=Heading(title='Pearson correlations', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x132fbbc50>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x132fbaa90>, _source_notes=['Pearson correlations above the diagonal, Spearman rank correlations below. Observations: 333.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x132fba850>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, category='table', type='

Brighter municipalities were less poor. Across the 333 units with lights, the correlation between lights per capita and the MPI in 2012 is -0.54, so lights alone account for 29% of the variation in poverty. An increase of one log point in lights per capita goes with an MPI 0.033 lower.

Three municipalities are almost unlit (Mojinete in Potosí, San Lorenzo in Pando and Exaltación in Beni). They do not drive the result: without them the correlation is -0.57, and the rank correlation is -0.60.

The relation carries over to 2024 almost as strongly: the lights of 2012 correlate -0.49 with poverty in 2024, and the lights of 2020 -0.50. Brighter municipalities saw smaller falls in poverty (correlation 0.50 between the lights of 2012 and the change), largely because they had less poverty to lose. Lights themselves rose in 95% of the units between 2012 and 2020, most in Beni and least in Santa Cruz.

- How heterogeneous is the relationship between poverty and night-time lights across departments?

`geo.scatter()` draws one panel, so the small multiples are Plotly Express; the slope and the correlation of each department come from `geo.scatter()` on that department's municipalities.

In [34]:
px.scatter(d12, x="ln_t400NTLpc", y="sdg1_2_mpi", facet_col="department",
           facet_col_wrap=3, trendline="ols", hover_name="name", height=700,
           labels={"ln_t400NTLpc": "Log lights per capita", "sdg1_2_mpi": "MPI",
                   "department": "Department"},
           title="Night-time lights and poverty by department, 2012")

In [35]:
rows = []
for department, d in d12.groupby("department"):
    fit = geo.scatter(d, "ln_t400NTLpc", "sdg1_2_mpi")
    rows.append({"department": department, "municipalities": fit.n,
                 "slope": fit.slope, "correlation": fit.corr})
potosi = d12[(d12["department"] == "Potosí") & (d12["name"] != "Mojinete")]
fit = geo.scatter(potosi, "ln_t400NTLpc", "sdg1_2_mpi")
print(f"Potosí without Mojinete: slope {fit.slope:.3f}, correlation {fit.corr:.2f}")
pd.DataFrame(rows).sort_values("slope").round(3)

Potosí without Mojinete: slope -0.058, correlation -0.74


,department,municipalities,slope,correlation
2,Cochabamba,46,-0.085,-0.791
1,Chuquisaca,29,-0.068,-0.849
4,Oruro,35,-0.045,-0.518
6,Potosí,37,-0.026,-0.536
7,Santa Cruz,56,-0.025,-0.365
3,La Paz,87,-0.024,-0.426
8,Tarija,11,-0.015,-0.429
5,Pando,15,-0.014,-0.321
0,Beni,17,-0.013,-0.465


The relationship is negative in all nine departments, but its strength varies. It is steepest and tightest in Cochabamba (slope -0.085, correlation -0.79) and Chuquisaca (-0.068, -0.85), where lights separate the poorer rural municipalities from the rest well. It is weakest in Pando (correlation -0.321) and Santa Cruz (-0.365), and flattest in Beni, Pando and Tarija (slopes between -0.013 and -0.015); in Tarija poverty is uniformly low. Potosí's slope (-0.026) is flattened by Mojinete, its almost unlit outlier: without it the slope is -0.058 and the correlation -0.74.

## Summary

- Multidimensional poverty fell in all 343 municipalities and indigenous autonomies between 2012 and 2024: the municipal mean went from 0.234 to 0.063 (median 0.227 to 0.053), and the population-weighted national mean from 0.133 to 0.033.
- In both periods poverty varies much more within departments than between them (about a fifth of the Theil index lies between departments). Chuquisaca, Pando and Potosí had the poorest municipalities in 2012; Chuquisaca, Potosí and Beni in 2024. Tarija and Santa Cruz are the least poor in both. TIOC-Raqaypampa is the poorest unit and Nuestra Señora de La Paz the least poor in both periods.
- Poorer municipalities fell most in absolute terms and the ordering persisted (correlation 0.86): absolute gaps narrowed, relative gaps widened (coefficient of variation 0.45 to 0.70).
- Larger municipalities are less poor (correlation with log population -0.34 in 2012, -0.23 in 2024). The large cities have low poverty and hold close to half the population, so the population-weighted MPI is about half the municipal average.
- Night-time lights per capita are more closely associated with poverty (correlation -0.54 in 2012, on the 333 units with lights). The association is robust to the few almost unlit municipalities, negative in every department, and still -0.50 between the lights of 2020 and poverty in 2024.
- The maps show that poverty and lights are spatially clustered. The companion notebooks [`esda`](https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/esda.ipynb) and [`esda_dependence`](https://colab.research.google.com/github/quarcs-lab/gdo4bolivia/blob/main/notebooks/esda_dependence.ipynb) measure that clustering with spatial weights, Moran's I and LISA.
- These are associations between variables measured in the same places; none of them shows that one causes the other. The [package documentation](https://quarcs-lab.github.io/geo4bolivia/) has every function used here, with an example.